# Retrieval Example Grids

This notebook builds qualitative retrieval figures from the trained EEG, MEG, and fMRI models.

It produces two figure styles:

- per-modality decoding galleries, where each row is a stimulus image and its top retrieved images
- a shared-stimulus cross-modality comparison, where the same stimulus image is compared across EEG, MEG, and fMRI

Generated figures are saved to `../results/summary/figures/`.

In [1]:
import copy
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from PIL import Image, ImageOps

sys.path.insert(0, os.path.abspath('..'))

from src.data.image_manifest import default_intersection_manifest_path
from src.eval_utils import (
    build_model,
    clip_embeddings_for_ids,
    collect_modality_embeddings,
    create_dataloader,
    load_checkpoint,
    load_clip_cache,
)

with open('../config.yaml', 'r') as handle:
    CONFIG = yaml.safe_load(handle)

ROOT = Path('..').resolve()
THINGS_ROOT = ROOT / CONFIG['data']['things_image_root']
EEG_STIMULI_ROOT = ROOT / CONFIG['data']['eeg_stimuli_dir']
UNION_MANIFEST = ROOT / 'data' / 'manifests' / 'all_modalities_union.tsv'
FIGURE_DIR = ROOT / 'results' / 'summary' / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

CONFIG_ABS = copy.deepcopy(CONFIG)
for key in [
    'eeg_dir',
    'eeg_stimuli_dir',
    'manifests_dir',
    'things_image_map_path',
    'things_image_root',
    'meg_dir',
    'fmri_dir',
    'clip_cache_dir',
]:
    if key in CONFIG_ABS['data']:
        CONFIG_ABS['data'][key] = str(ROOT / CONFIG_ABS['data'][key])

DEVICE = (
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)
print(f'Using device: {DEVICE}')

Using device: mps


In [2]:
IMAGE_DF = pd.read_csv(UNION_MANIFEST, sep='\t')
IMAGE_DF = IMAGE_DF.drop_duplicates(subset=['image_id']).reset_index(drop=True)
IMAGE_PATHS = dict(zip(IMAGE_DF['image_id'], IMAGE_DF['relative_path']))

def resolve_image_path(image_id: str) -> Path:
    relative_path = IMAGE_PATHS.get(image_id)
    if relative_path:
        things_path = THINGS_ROOT / relative_path
        if things_path.exists():
            return things_path

    eeg_path = EEG_STIMULI_ROOT / f'{image_id}.jpg'
    if eeg_path.exists():
        return eeg_path

    raise FileNotFoundError(f'Could not resolve image path for image_id={image_id!r}')


def load_display_image(image_id: str, size=(224, 224)):
    path = resolve_image_path(image_id)
    image = Image.open(path).convert('RGB')
    image = ImageOps.fit(image, size, method=Image.Resampling.LANCZOS)
    return np.asarray(image)


IMAGE_DF.head()

,image_id,relative_path,source,image_number
0,aardvark_01b,aardvark/aardvark_01b.jpg,"eeg,meg",1.0
1,aardvark_02s,aardvark/aardvark_02s.jpg,"eeg,meg",2.0
2,aardvark_03s,aardvark/aardvark_03s.jpg,"eeg,meg",3.0
3,aardvark_04s,aardvark/aardvark_04s.jpg,"eeg,meg",4.0
4,aardvark_05s,aardvark/aardvark_05s.jpg,"eeg,meg",5.0


## Configuration

Adjust the subject IDs, number of examples, and figure sizes here if needed.

In [3]:
SUBJECTS = {
    'eeg': 1,
    'meg': 1,
    'fmri': 1,
}

TOP_K = 5
PER_MODALITY_ROWS = 5
SHARED_EXAMPLE_ROWS = 2
RANDOM_SEED = 7

MODALITY_ORDER = ['eeg', 'meg', 'fmri']
STATE_CACHE = {}

np.random.seed(RANDOM_SEED)

In [4]:
def infer_checkpoint_path(modality: str, subject: int, shared_only: bool = False) -> Path:
    stem = f'{modality}_brainalign_sub{subject:02d}'
    if modality == 'meg':
        stem += '_attnpool'
    if shared_only:
        stem += '_shared'
    return ROOT / 'checkpoints' / modality / f'{stem}_best.pt'


def build_state_key(modality: str, subject: int, split: str, shared_only: bool, shared_manifest: str | None):
    return (modality, subject, split, shared_only, str(shared_manifest) if shared_manifest else None)


def collect_retrieval_state(
    modality: str,
    subject: int,
    split: str = 'test',
    shared_only: bool = False,
    shared_manifest: str | None = None,
    checkpoint_path: str | Path | None = None,
):
    key = build_state_key(modality, subject, split, shared_only, shared_manifest)
    if key in STATE_CACHE:
        return STATE_CACHE[key]

    config = copy.deepcopy(CONFIG_ABS)
    if shared_manifest:
        config.setdefault('data', {})['shared_manifest_path'] = shared_manifest

    checkpoint_path = Path(checkpoint_path) if checkpoint_path else infer_checkpoint_path(modality, subject, shared_only)
    if not checkpoint_path.exists():
        raise FileNotFoundError(f'Checkpoint not found: {checkpoint_path}')

    clip_dict = load_clip_cache(config)
    loader = create_dataloader(
        config,
        modality,
        split,
        subject=subject,
        shared_only=shared_only,
        quiet=False,
        shuffle=False,
    )
    sample_x = loader.dataset[0]['x']
    model = build_model(config, modality, sample_x, DEVICE)
    load_checkpoint(model, str(checkpoint_path), DEVICE)

    averaged_embeddings = collect_modality_embeddings(model, loader, DEVICE)
    image_ids = sorted(averaged_embeddings)
    modality_matrix = np.stack([averaged_embeddings[image_id] for image_id in image_ids], axis=0).astype(np.float32)
    clip_matrix = clip_embeddings_for_ids(clip_dict, image_ids)
    similarity = modality_matrix @ clip_matrix.T
    rankings = np.argsort(similarity, axis=1)[:, ::-1]

    records = []
    for row_idx, image_id in enumerate(image_ids):
        ranked_ids = [image_ids[col_idx] for col_idx in rankings[row_idx, :TOP_K]]
        ranked_scores = [float(similarity[row_idx, col_idx]) for col_idx in rankings[row_idx, :TOP_K]]
        records.append(
            {
                'image_id': image_id,
                'top_ids': ranked_ids,
                'top_scores': ranked_scores,
                'top1_correct': ranked_ids[0] == image_id,
                'self_rank': int(np.where(rankings[row_idx] == row_idx)[0][0]) + 1,
            }
        )

    state = {
        'modality': modality,
        'subject': subject,
        'split': split,
        'shared_only': shared_only,
        'shared_manifest': shared_manifest,
        'checkpoint_path': str(checkpoint_path),
        'records': records,
        'record_lookup': {record['image_id']: record for record in records},
        'image_ids': image_ids,
    }
    STATE_CACHE[key] = state
    return state

In [5]:
def choose_example_ids(state, n_rows=5, prefer_correct=True):
    records = state['records']
    if prefer_correct:
        pool = [record for record in records if record['top1_correct']]
    else:
        pool = records

    if len(pool) < n_rows:
        pool = records

    step = max(1, len(pool) // n_rows)
    selected = [pool[idx]['image_id'] for idx in range(0, len(pool), step)][:n_rows]
    return selected


def choose_shared_example_ids(states_by_modality, n_rows=2, prefer_correct_any=True):
    common = set.intersection(*(set(state['image_ids']) for state in states_by_modality.values()))
    common = sorted(common)
    if not common:
        raise ValueError('No shared image IDs available across the selected modalities')

    scored = []
    for image_id in common:
        score = 0
        for state in states_by_modality.values():
            record = state['record_lookup'][image_id]
            score += int(record['top1_correct'])
        scored.append((score, image_id))

    if prefer_correct_any:
        scored.sort(key=lambda item: (-item[0], item[1]))
    else:
        scored.sort(key=lambda item: item[1])

    return [image_id for _, image_id in scored[:n_rows]]

In [6]:
def draw_image(ax, image_id: str, title: str | None = None):
    ax.imshow(load_display_image(image_id))
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    if title:
        ax.set_title(title, fontsize=10)


def plot_modality_gallery(states_by_modality, example_ids_by_modality, top_k=5, save_path: Path | None = None):
    total_rows = sum(len(example_ids_by_modality[modality]) for modality in MODALITY_ORDER)
    n_cols = top_k + 1
    fig, axes = plt.subplots(total_rows, n_cols, figsize=(2.6 * n_cols, 2.2 * total_rows), constrained_layout=True)
    if total_rows == 1:
        axes = np.expand_dims(axes, axis=0)
    if n_cols == 1:
        axes = np.expand_dims(axes, axis=1)

    row_offset = 0
    modality_centers = {}
    for modality in MODALITY_ORDER:
        state = states_by_modality[modality]
        example_ids = example_ids_by_modality[modality]
        start_row = row_offset
        for local_row, image_id in enumerate(example_ids):
            row_idx = row_offset + local_row
            record = state['record_lookup'][image_id]
            draw_image(axes[row_idx, 0], image_id, title='Stimulus' if row_idx == 0 else None)
            for col_idx, retrieved_id in enumerate(record['top_ids'][:top_k], start=1):
                title = f'Top-{col_idx}' if row_idx == 0 else None
                draw_image(axes[row_idx, col_idx], retrieved_id, title=title)
        end_row = row_offset + len(example_ids) - 1
        modality_centers[modality] = (start_row + end_row) / 2
        row_offset += len(example_ids)

    for modality, center_row in modality_centers.items():
        y = 1 - ((center_row + 0.5) / total_rows)
        fig.text(0.01, y, modality.upper(), fontsize=18, va='center', ha='left')

    fig.suptitle('Qualitative retrieval examples by modality', fontsize=18)
    if save_path is not None:
        fig.savefig(save_path, dpi=200, bbox_inches='tight')
        print(f'Saved {save_path}')
    plt.show()


def plot_shared_stimulus_comparison(states_by_modality, shared_image_ids, top_k=5, save_path: Path | None = None):
    total_rows = len(MODALITY_ORDER) * len(shared_image_ids)
    n_cols = top_k + 1
    fig, axes = plt.subplots(total_rows, n_cols, figsize=(2.6 * n_cols, 2.2 * total_rows), constrained_layout=True)
    if total_rows == 1:
        axes = np.expand_dims(axes, axis=0)
    if n_cols == 1:
        axes = np.expand_dims(axes, axis=1)

    row_idx = 0
    modality_centers = {modality: [] for modality in MODALITY_ORDER}
    for modality in MODALITY_ORDER:
        state = states_by_modality[modality]
        for image_id in shared_image_ids:
            record = state['record_lookup'][image_id]
            draw_image(axes[row_idx, 0], image_id, title='Encoded img' if row_idx == 0 else None)
            for col_idx, retrieved_id in enumerate(record['top_ids'][:top_k], start=1):
                title = 'Retrieved images' if row_idx == 0 and col_idx == 1 else None
                draw_image(axes[row_idx, col_idx], retrieved_id, title=title)
            modality_centers[modality].append(row_idx)
            row_idx += 1

    for modality, rows in modality_centers.items():
        y = 1 - (((min(rows) + max(rows)) / 2 + 0.5) / total_rows)
        fig.text(0.01, y, modality.upper(), fontsize=18, va='center', ha='left')

    fig.suptitle('Shared-stimulus retrieval comparison across modalities', fontsize=18)
    if save_path is not None:
        fig.savefig(save_path, dpi=200, bbox_inches='tight')
        print(f'Saved {save_path}')
    plt.show()

## Native test-split retrieval examples

This section uses each modality's native `test` split and selected subject.

In [ ]:
native_states = {
    modality: collect_retrieval_state(modality, subject, split='test', shared_only=False)
    for modality, subject in SUBJECTS.items()
}

native_example_ids = {
    modality: choose_example_ids(state, n_rows=PER_MODALITY_ROWS, prefer_correct=True)
    for modality, state in native_states.items()
}

native_example_ids

Loading CLIP cache from /Users/thomas/Documents/Projects/brainalign_ext_meg_fmri/clip_cache/ViT-B-32.npz
Loading EEG test data for subject 01...
Loaded 200 conditions with 80 repetitions each.
Loading pretrained CBraMod weights from: /Users/thomas/Documents/Projects/brainalign_ext_meg_fmri/src/vendored/CBraMod/pretrained_weights/pretrained_weights.pth


/Users/thomas/Documents/Projects/brainalign_ext_meg_fmri/src/vendored/CBraMod/models/cbramod.py:84: UserWarning: An output with one or more elements was resized since it had shape [], which does not match the required output shape [2016, 101]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/Resize.cpp:38.)
  spectral = torch.fft.rfft(mask_x, dim=-1, norm='forward')


Loading CLIP cache from /Users/thomas/Documents/Projects/brainalign_ext_meg_fmri/clip_cache/ViT-B-32.npz
Using merged/root MEG epochs file for Subject 1: preprocessed_P1-epo.fif (split shards are not loaded separately).
Reading preprocessed_P1-epo.fif...


In [ ]:
plot_modality_gallery(
    native_states,
    native_example_ids,
    top_k=TOP_K,
    save_path=FIGURE_DIR / 'retrieval_examples_by_modality.png',
)

NameError: name 'native_states' is not defined

## Shared-stimulus comparison across EEG, MEG, and fMRI

This section restricts all three modalities to the shared three-way image intersection and compares the same stimulus images across modalities.

In [ ]:
shared_manifest = default_intersection_manifest_path(CONFIG_ABS, ['eeg', 'meg', 'fmri'])
assert shared_manifest.exists(), f'Missing 3-way shared manifest: {shared_manifest}'
shared_manifest = str(shared_manifest)

shared_states = {
    modality: collect_retrieval_state(
        modality,
        subject,
        split='test',
        shared_only=True,
        shared_manifest=shared_manifest,
    )
    for modality, subject in SUBJECTS.items()
}

shared_example_ids = choose_shared_example_ids(shared_states, n_rows=SHARED_EXAMPLE_ROWS, prefer_correct_any=True)
shared_example_ids

FileNotFoundError: Checkpoint not found: /Users/thomas/Documents/Projects/brainalign_ext_meg_fmri/checkpoints/eeg/eeg_brainalign_sub01_shared_best.pt

In [ ]:
plot_shared_stimulus_comparison(
    shared_states,
    shared_example_ids,
    top_k=TOP_K,
    save_path=FIGURE_DIR / 'retrieval_examples_shared_stimuli.png',
)

NameError: name 'shared_states' is not defined

In [ ]:
for modality, state in native_states.items():
    correct_rate = np.mean([record['top1_correct'] for record in state['records']]) * 100
    print(f"{modality.upper()} sub-{state['subject']:02d}: {len(state['records'])} averaged image embeddings, top-1 on image-level retrieval = {correct_rate:.2f}%")

NameError: name 'native_states' is not defined